# H1 · Visual encoder có giữ thông tin màu áo không?

Notebook độc lập cho `detr-finetune-random-resize.ipynb`. Dùng **cùng REC checkpoint với H2**, chạy ảnh qua nhánh thị giác đã fine-tune ở chế độ eval, lấy spatial memory 256 kênh **trước `image_projection` và fusion**. Pool feature theo bbox ground truth từng người, rồi huấn luyện một lớp tuyến tính dự đoán màu áo.

1. Lưu notebook trên Google Drive và mở bằng Google Colab. Chạy cell mount Drive. Notebook tìm thư mục chứa file theo tên; nếu m đổi tên hoặc có nhiều bản trùng tên, điền `NOTEBOOK_DIR` trong cell đó. Đặt hai checkpoint cạnh notebook hoặc sửa đường dẫn trong cell cấu hình. RefCOCO tự tải vào thư mục con `refcoco_dataset`.
2. Tự tạo nhãn màu áo cho train và val từ mô tả RefCOCO.
3. Trích và lưu cache. Có thể dừng tại đây nếu thiếu thời gian.
4. Probe mặc định chạy sau bước trích feature. Dùng GPU để trích feature.

Các file `color_labels_auto.csv`, `color_features_auto.pt`, `probe_metrics_auto.json` và `probe_val_predictions_auto.csv` được lưu cạnh notebook trên Drive. Khi mở phiên Colab mới, chạy lại cell mount để dùng tiếp dữ liệu đã lưu.


In [1]:
from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/Computer Vision/kiểm chứng')
NOTEBOOK_FILENAME = 'H1_color_linear_probe_auto.ipynb'
NOTEBOOK_DIR = None  # Ví dụ: DRIVE_ROOT / 'CV' / 'H1'

if NOTEBOOK_DIR is None:
    matches = sorted(DRIVE_ROOT.rglob(NOTEBOOK_FILENAME))
    if len(matches) != 1:
        found = '\n'.join(str(path) for path in matches[:10])
        raise RuntimeError(
            f'Tìm được {len(matches)} file tên {NOTEBOOK_FILENAME}. '
            f'Điền NOTEBOOK_DIR bằng đường dẫn thư mục chứa notebook.\n{found}'
        )
    NOTEBOOK_DIR = matches[0].parent
else:
    NOTEBOOK_DIR = Path(NOTEBOOK_DIR)
    if not NOTEBOOK_DIR.is_absolute():
        NOTEBOOK_DIR = DRIVE_ROOT / NOTEBOOK_DIR

if not NOTEBOOK_DIR.is_dir():
    raise FileNotFoundError(f'Không tìm thấy thư mục notebook: {NOTEBOOK_DIR}')
print('Thư mục notebook:', NOTEBOOK_DIR)

Thư mục notebook: /content/drive/MyDrive/Computer Vision/kiểm chứng


In [3]:
import inspect
import subprocess
import sys

DATASET_HANDLE = 'lamky690/refcoco/versions/1'
DATASET_DIR = NOTEBOOK_DIR / 'refcoco_dataset'

def locate_refcoco(root):
    matches = [annotation for annotation in root.rglob('person_samples.json')
               if (annotation.parent / 'images').is_dir()
               and next((annotation.parent / 'images').iterdir(), None) is not None]
    if len(matches) > 1:
        raise RuntimeError(f'Có nhiều bản RefCOCO trong {root}: {matches}')
    return matches[0] if matches else None

ANNOTATION_PATH = locate_refcoco(DATASET_DIR)
if ANNOTATION_PATH is None:
    try:
        import kagglehub
    except ImportError:
        kagglehub = None
    if kagglehub is None or 'output_dir' not in inspect.signature(kagglehub.dataset_download).parameters:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'kagglehub'])
        if kagglehub is not None:
            raise RuntimeError('Đã cập nhật kagglehub. Restart session Colab rồi chạy lại từ đầu.')
        import kagglehub

    DATASET_DIR.mkdir(exist_ok=True)
    downloaded = Path(kagglehub.dataset_download(
        DATASET_HANDLE, output_dir=str(DATASET_DIR)))
    ANNOTATION_PATH = locate_refcoco(downloaded)
    if ANNOTATION_PATH is None:
        raise FileNotFoundError(f'Thiếu person_samples.json hoặc thư mục images trong {downloaded}.')

IMAGE_DIR = ANNOTATION_PATH.parent / 'images'
print('Annotation:', ANNOTATION_PATH)
print('Ảnh:', IMAGE_DIR)

Annotation: /content/drive/MyDrive/Computer Vision/kiểm chứng/refcoco_dataset/refcoco_data/person_samples.json
Ảnh: /content/drive/MyDrive/Computer Vision/kiểm chứng/refcoco_dataset/refcoco_data/images


In [4]:
from pathlib import Path
import hashlib
import json
import math
import random
import re
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torchvision import transforms
from torchvision.ops import roi_align
from tqdm.auto import tqdm
from IPython.display import display

DETR_CHECKPOINT = NOTEBOOK_DIR / 'detr-r50-unc.pth'
REC_CHECKPOINT = NOTEBOOK_DIR / 'best_checkpoint.pth'  # Checkpoint REC đã dùng cho H2
OUT = NOTEBOOK_DIR

LABEL_PATH = OUT / 'color_labels_auto.csv'
CACHE_PATH = OUT / 'color_features_auto.pt'
COLORS = ('red', 'blue', 'white', 'black', 'green', 'yellow')
CANDIDATES_PER_COLOR = {'train': 70, 'val': 25}
SEED = 42
BATCH_IMAGES = 8
EXTRACT_FEATURES = True
RUN_PROBE = True
MIN_TRAIN_PER_COLOR = 10
MIN_VAL_PER_COLOR = 5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def sha256(path):
    with path.open('rb') as stream:
        digest = hashlib.sha256()
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
        return digest.hexdigest()

def save_labels(frame):
    temporary = LABEL_PATH.with_suffix('.tmp')
    frame.to_csv(temporary, index=False)
    temporary.replace(LABEL_PATH)

## 1. Tạo nhãn màu áo từ mô tả

Bỏ cả câu khi có từ chỉ vị trí như `left`, `right`, `next to`, `beside` hoặc `on the side`. Tìm cụm màu đi cùng từ chỉ áo, chẳng hạn `black shirt`. Nếu các câu còn lại gợi ý nhiều màu khác nhau cho cùng một người thì bỏ người đó. Cả train và val dùng nhãn suy ra từ text.

In [5]:
samples = json.loads(ANNOTATION_PATH.read_text(encoding='utf-8'))
annotation_sha = sha256(ANNOTATION_PATH)
if not LABEL_PATH.exists():
    words = '|'.join(map(re.escape, COLORS))
    pattern = re.compile(
        rf"\b({words})\s+(?:(?:striped|long-sleeve|short-sleeve)\s+)?"
        r"(?:t[ -]?shirt|shirt|jacket|hoodie|sweater|top|jersey|blouse|coat)\b",
        re.IGNORECASE,
    )
    position_pattern = re.compile(
        r"\b(?:left(?:most|[\s-]+hand)?|right(?:most|[\s-]+hand)?|"
        r"next[\s-]+to|close[\s-]+to|beside|alongside|adjacent[\s-]+to|"
        r"near(?:est|by)?|behind|in[\s-]+front[\s-]+of|"
        r"between|across[\s-]+from|on[\s-]+the[\s-]+side|middle|center)\b",
        re.IGNORECASE,
    )
    people = defaultdict(list)
    for sample in samples:
        if sample['split'] in CANDIDATES_PER_COLOR:
            people[(sample['image_id'], sample['ann_id'])].append(sample)

    groups = defaultdict(list)
    for (image_id, ann_id), references in people.items():
        matches = [
            (sample, color.lower())
            for sample in references
            if not position_pattern.search(sample['text'])
            for color in pattern.findall(sample['text'])
        ]
        if len({color for _, color in matches}) != 1:
            continue
        sample, color = matches[0]
        x, y, w, h = sample['bbox']
        groups[sample['split'], color].append(dict(
            review_id=f'{image_id}:{ann_id}', split=sample['split'],
            image_id=image_id, file_name=sample['file_name'],
            width=sample['width'], height=sample['height'],
            x=x, y=y, w=w, h=h, color=color, text=sample['text'], annotation_sha256=annotation_sha,
        ))

    rng = random.Random(SEED)
    selected = []
    for (split, color), group in sorted(groups.items()):
        selected.extend(rng.sample(group, min(len(group), CANDIDATES_PER_COLOR[split])))
    if not selected:
        raise ValueError('Không tìm được câu gợi ý màu áo. Kiểm tra dữ liệu và COLORS.')
    save_labels(pd.DataFrame(selected))

def read_labels():
    frame = pd.read_csv(LABEL_PATH, keep_default_na=False)
    if frame.empty or set(frame['annotation_sha256']) != {annotation_sha}:
        raise ValueError('CSV thuộc annotation khác. Dùng OUT khác để tạo bảng mới.')
    if frame['review_id'].duplicated().any():
        raise ValueError('CSV có review_id trùng.')
    frame['color'] = frame['color'].str.strip().str.lower()
    invalid = set(frame['color']) - set(COLORS)
    if invalid:
        raise ValueError(f'Màu chưa có trong COLORS: {invalid}')
    return frame

labels_frame = read_labels()
display(pd.crosstab(labels_frame['split'], labels_frame['color']))
print(LABEL_PATH)

color,black,blue,green,red,white,yellow
split,,,,,,
train,70,70,70,70,70,70
val,25,25,25,25,25,25


/content/drive/MyDrive/Computer Vision/kiểm chứng/color_labels_auto.csv


## 2. Định nghĩa visual encoder

Cell dưới giữ các lớp DETR từ notebook gốc. Stem đến layer3 lấy từ checkpoint DETR gốc. Các phần `layer4`, `proj` và `visual_encoder` lấy trọng số đã fine-tune từ REC checkpoint.

In [6]:
# Copyright (c) Facebook, Inc. and its affiliates. All Rights Reserved
# Visual modules from djiajunustc/TransVG, commit:
# c8624277018fc9786a39181e229f155f933b2b59
# https://github.com/djiajunustc/TransVG/tree/main/models/visual_model
# Minimal NestedTensor utility + official backbone/PE/encoder-only execution.
import copy
import argparse
import pickle
import torchvision
from typing import Dict, List, Optional
from torch import Tensor
from torchvision.models._utils import IntermediateLayerGetter

class NestedTensor:
    def __init__(self,tensors,mask):
        self.tensors, self.mask = tensors, mask
    def decompose(self): return self.tensors,self.mask

def nested_tensor_from_tensor_list(images):
    # Tất cả ảnh trong batch cùng H,W; không padding.
    assert isinstance(images,torch.Tensor) and images.ndim == 4
    mask = torch.zeros(images.shape[0],*images.shape[-2:],device=images.device,dtype=torch.bool)
    return NestedTensor(images,mask)

class PositionEmbeddingSine(nn.Module):
    """
    This is a more standard version of the position embedding, very similar to the one
    used by the Attention is all you need paper, generalized to work on images.
    """
    def __init__(self, num_pos_feats=64, temperature=10000, normalize=False, scale=None):
        super().__init__()
        self.num_pos_feats = num_pos_feats
        self.temperature = temperature
        self.normalize = normalize
        if scale is not None and normalize is False:
            raise ValueError("normalize should be True if scale is passed")
        if scale is None:
            scale = 2 * math.pi
        self.scale = scale

    def forward(self, tensor_list: NestedTensor):
        x = tensor_list.tensors
        mask = tensor_list.mask
        assert mask is not None
        not_mask = ~mask
        y_embed = not_mask.cumsum(1, dtype=torch.float32)
        x_embed = not_mask.cumsum(2, dtype=torch.float32)
        if self.normalize:
            eps = 1e-6
            y_embed = y_embed / (y_embed[:, -1:, :] + eps) * self.scale
            x_embed = x_embed / (x_embed[:, :, -1:] + eps) * self.scale

        dim_t = torch.arange(self.num_pos_feats, dtype=torch.float32, device=x.device)
        dim_t = self.temperature ** (2 * (dim_t // 2) / self.num_pos_feats)

        pos_x = x_embed[:, :, :, None] / dim_t
        pos_y = y_embed[:, :, :, None] / dim_t
        pos_x = torch.stack((pos_x[:, :, :, 0::2].sin(), pos_x[:, :, :, 1::2].cos()), dim=4).flatten(3)
        pos_y = torch.stack((pos_y[:, :, :, 0::2].sin(), pos_y[:, :, :, 1::2].cos()), dim=4).flatten(3)
        pos = torch.cat((pos_y, pos_x), dim=3).permute(0, 3, 1, 2)
        return pos

class FrozenBatchNorm2d(torch.nn.Module):
    """
    BatchNorm2d where the batch statistics and the affine parameters are fixed.

    Copy-paste from torchvision.misc.ops with added eps before rqsrt,
    without which any other models than torchvision.models.resnet[18,34,50,101]
    produce nans.
    """

    def __init__(self, n):
        super(FrozenBatchNorm2d, self).__init__()
        self.register_buffer("weight", torch.ones(n))
        self.register_buffer("bias", torch.zeros(n))
        self.register_buffer("running_mean", torch.zeros(n))
        self.register_buffer("running_var", torch.ones(n))

    def _load_from_state_dict(self, state_dict, prefix, local_metadata, strict,
                              missing_keys, unexpected_keys, error_msgs):
        num_batches_tracked_key = prefix + 'num_batches_tracked'
        if num_batches_tracked_key in state_dict:
            del state_dict[num_batches_tracked_key]

        super(FrozenBatchNorm2d, self)._load_from_state_dict(
            state_dict, prefix, local_metadata, strict,
            missing_keys, unexpected_keys, error_msgs)

    def forward(self, x):
        # move reshapes to the beginning
        # to make it fuser-friendly
        w = self.weight.reshape(1, -1, 1, 1)
        b = self.bias.reshape(1, -1, 1, 1)
        rv = self.running_var.reshape(1, -1, 1, 1)
        rm = self.running_mean.reshape(1, -1, 1, 1)
        eps = 1e-5
        scale = w * (rv + eps).rsqrt()
        bias = b - rm * scale
        return x * scale + bias

class BackboneBase(nn.Module):

    def __init__(self, name:str, backbone: nn.Module, num_channels: int, return_interm_layers: bool):
        super().__init__()
        for name, parameter in backbone.named_parameters():
            if 'layer2' not in name and 'layer3' not in name and 'layer4' not in name:
                parameter.requires_grad_(False)
        if return_interm_layers:
            return_layers = {"layer1": "0", "layer2": "1", "layer3": "2", "layer4": "3"}
        else:
            return_layers = {'layer4': "0"}
        self.body = IntermediateLayerGetter(backbone, return_layers=return_layers)
        self.num_channels = num_channels

    def forward(self, tensor_list: NestedTensor):
        xs = self.body(tensor_list.tensors)
        out: Dict[str, NestedTensor] = {}
        for name, x in xs.items():
            m = tensor_list.mask
            assert m is not None
            mask = F.interpolate(m[None].float(), size=x.shape[-2:]).to(torch.bool)[0]
            out[name] = NestedTensor(x, mask)
        return out

class Backbone(BackboneBase):
    """ResNet backbone with frozen BatchNorm."""
    def __init__(self, name: str,
                 return_interm_layers: bool,
                 dilation: bool):

        backbone = getattr(torchvision.models, name)(
            replace_stride_with_dilation=[False, False, dilation],
            weights=None, norm_layer=FrozenBatchNorm2d)
            # pretrained=is_main_process(), norm_layer=FrozenBatchNorm2d)
        assert name in ('resnet50', 'resnet101')
        num_channels = 2048
        super().__init__(name, backbone, num_channels, return_interm_layers)

class Joiner(nn.Sequential):
    def __init__(self, backbone, position_embedding):
        super().__init__(backbone, position_embedding)

    def forward(self, tensor_list: NestedTensor):
        xs = self[0](tensor_list)
        out: List[NestedTensor] = []
        pos = []
        for name, x in xs.items():
            out.append(x)
            # position encoding
            pos.append(self[1](x).to(x.tensors.dtype))

        return out, pos

class Transformer(nn.Module):

    def __init__(self, d_model=512, nhead=8, num_encoder_layers=6,
                 num_decoder_layers=6, dim_feedforward=2048, dropout=0.1,
                 activation="relu", normalize_before=False,
                 return_intermediate_dec=False):
        super().__init__()

        encoder_layer = TransformerEncoderLayer(d_model, nhead, dim_feedforward,
                                                dropout, activation, normalize_before)
        encoder_norm = nn.LayerNorm(d_model) if normalize_before else None
        self.encoder = TransformerEncoder(encoder_layer, num_encoder_layers, encoder_norm)

        if num_decoder_layers > 0:
            decoder_layer = TransformerDecoderLayer(d_model, nhead, dim_feedforward,
                                                    dropout, activation, normalize_before)
            decoder_norm = nn.LayerNorm(d_model)
            self.decoder = TransformerDecoder(decoder_layer, num_decoder_layers, decoder_norm,
                                            return_intermediate=return_intermediate_dec)
        else:
            self.decoder = None

        self._reset_parameters()

        self.d_model = d_model
        self.nhead = nhead

    def _reset_parameters(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, src, mask, pos_embed, query_embed=None):
        # flatten NxCxHxW to HWxNxC
        bs, c, h, w = src.shape
        src = src.flatten(2).permute(2, 0, 1)
        pos_embed = pos_embed.flatten(2).permute(2, 0, 1)
        mask = mask.flatten(1)
        memory = self.encoder(src, src_key_padding_mask=mask, pos=pos_embed)

        if self.decoder is not None:
            query_embed = query_embed.unsqueeze(1).repeat(1, bs, 1)
            tgt = torch.zeros_like(query_embed)
            hs = self.decoder(tgt, memory, memory_key_padding_mask=mask,
                            pos=pos_embed, query_pos=query_embed)

            return hs.transpose(1, 2), memory.permute(1, 2, 0).view(bs, c, h, w)
        else:
            return mask, memory

class TransformerEncoder(nn.Module):

    def __init__(self, encoder_layer, num_layers, norm=None):
        super().__init__()
        self.layers = _get_clones(encoder_layer, num_layers)
        self.num_layers = num_layers
        self.norm = norm

    def forward(self, src,
                mask: Optional[Tensor] = None,
                src_key_padding_mask: Optional[Tensor] = None,
                pos: Optional[Tensor] = None):
        output = src

        for layer in self.layers:
            output = layer(output, src_mask=mask,
                           src_key_padding_mask=src_key_padding_mask, pos=pos)

        if self.norm is not None:
            output = self.norm(output)

        return output

class TransformerEncoderLayer(nn.Module):

    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1,
                 activation="relu", normalize_before=False):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        # Implementation of Feedforward model
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

        self.activation = _get_activation_fn(activation)
        self.normalize_before = normalize_before

    def with_pos_embed(self, tensor, pos: Optional[Tensor]):
        return tensor if pos is None else tensor + pos

    def forward_post(self,
                     src,
                     src_mask: Optional[Tensor] = None,
                     src_key_padding_mask: Optional[Tensor] = None,
                     pos: Optional[Tensor] = None):
        q = k = self.with_pos_embed(src, pos)
        src2 = self.self_attn(q, k, value=src, attn_mask=src_mask,
                              key_padding_mask=src_key_padding_mask)[0]
        src = src + self.dropout1(src2)
        src = self.norm1(src)
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src))))
        src = src + self.dropout2(src2)
        src = self.norm2(src)
        return src

    def forward_pre(self, src,
                    src_mask: Optional[Tensor] = None,
                    src_key_padding_mask: Optional[Tensor] = None,
                    pos: Optional[Tensor] = None):
        src2 = self.norm1(src)
        q = k = self.with_pos_embed(src2, pos)
        src2 = self.self_attn(q, k, value=src2, attn_mask=src_mask,
                              key_padding_mask=src_key_padding_mask)[0]
        src = src + self.dropout1(src2)
        src2 = self.norm2(src)
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src2))))
        src = src + self.dropout2(src2)
        return src

    def forward(self, src,
                src_mask: Optional[Tensor] = None,
                src_key_padding_mask: Optional[Tensor] = None,
                pos: Optional[Tensor] = None):
        if self.normalize_before:
            return self.forward_pre(src, src_mask, src_key_padding_mask, pos)
        return self.forward_post(src, src_mask, src_key_padding_mask, pos)

def _get_clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for i in range(N)])

def _get_activation_fn(activation):
    """Return an activation function given a string"""
    if activation == "relu":
        return F.relu
    if activation == "gelu":
        return F.gelu
    if activation == "glu":
        return F.glu
    raise RuntimeError(F"activation should be relu/gelu, not {activation}.")

class DETR(nn.Module):
    """ This is the DETR module that performs object detection """
    def __init__(self, backbone, transformer, num_queries, train_backbone, train_transformer, aux_loss=False):
        """ Initializes the model.
        Parameters:
            backbone: torch module of the backbone to be used. See backbone.py
            transformer: torch module of the transformer architecture. See transformer.py
            num_classes: number of object classes
            num_queries: number of object queries, ie detection slot. This is the maximal number of objects
                         DETR can detect in a single image. For COCO, we recommend 100 queries.
            aux_loss: True if auxiliary decoding losses (loss at each decoder layer) are to be used.
        """
        super().__init__()
        self.num_queries = num_queries
        self.transformer = transformer
        self.backbone = backbone

        if self.transformer is not None:
            hidden_dim = transformer.d_model
            self.input_proj = nn.Conv2d(backbone.num_channels, hidden_dim, kernel_size=1)
        else:
            hidden_dim = backbone.num_channels

        if not train_backbone:
            for p in self.backbone.parameters():
                p.requires_grad_(False)

        if self.transformer is not None and not train_transformer:
            for m in [self.transformer, self.input_proj]:
                for p in m.parameters():
                    p.requires_grad_(False)

        self.num_channels = hidden_dim

    def forward(self, samples: NestedTensor):
        """ The forward expects a NestedTensor, which consists of:
               - samples.tensor: batched images, of shape [batch_size x 3 x H x W]
               - samples.mask: a binary mask of shape [batch_size x H x W], containing 1 on padded pixels

            It returns a dict with the following elements:
               - "pred_logits": the classification logits (including no-object) for all queries.
                                Shape= [batch_size x num_queries x (num_classes + 1)]
               - "pred_boxes": The normalized boxes coordinates for all queries, represented as
                               (center_x, center_y, height, width). These values are normalized in [0, 1],
                               relative to the size of each individual image (disregarding possible padding).
                               See PostProcess for information on how to retrieve the unnormalized bounding box.
               - "aux_outputs": Optional, only returned when auxilary losses are activated. It is a list of
                                dictionnaries containing the two above keys for each decoder layer.
        """
        if isinstance(samples, (list, torch.Tensor)):
            samples = nested_tensor_from_tensor_list(samples)
        features, pos = self.backbone(samples)

        src, mask = features[-1].decompose()
        assert mask is not None

        if self.transformer is not None:
            out = self.transformer(self.input_proj(src), mask, pos[-1], query_embed=None)
        else:
            out = [mask.flatten(1), src.flatten(2).permute(2, 0, 1)]

        return out

class FrozenDETRFeatures(nn.Module):
    def __init__(self,checkpoint):
        super().__init__()
        backbone = Backbone('resnet50', return_interm_layers=False, dilation=False)
        position = PositionEmbeddingSine(128, normalize=True)
        joiner = Joiner(backbone,position)
        joiner.num_channels = backbone.num_channels
        transformer = Transformer(d_model=256,nhead=8,num_encoder_layers=6,
                                  num_decoder_layers=0,dim_feedforward=2048,
                                  dropout=0.1,activation='relu',normalize_before=False)
        self.visual = DETR(joiner,transformer,num_queries=100,
                           train_backbone=False,train_transformer=False)
        # Một số checkpoint gốc chứa argparse.Namespace; chỉ allowlist lớp đó.
        with torch.serialization.safe_globals([argparse.Namespace]):
            payload = torch.load(checkpoint,map_location='cpu',weights_only=True)
        state = payload.get('model',payload)
        assert isinstance(state,dict) and state, 'Checkpoint thiếu model state_dict.'
        state = {k.removeprefix('module.'):v for k,v in state.items()}
        required = self.visual.state_dict()
        missing = sorted(set(required)-set(state))
        assert not missing, f'Checkpoint thiếu visual weights: {missing[:20]}'
        mismatched = [k for k in required if tuple(required[k].shape)!=tuple(state[k].shape)]
        assert not mismatched, f'Backbone/cấu hình checkpoint không khớp: {mismatched[:20]}'
        extra = sorted(set(state)-set(required))
        # Chỉ bỏ detection decoder/head/query; không bỏ nhầm encoder/backbone.
        allowed = ('transformer.decoder.','class_embed.','bbox_embed.','query_embed.')
        unexpected = [k for k in extra if not k.startswith(allowed)]
        assert not unexpected, f'Checkpoint có visual keys không nhận diện: {unexpected[:20]}'
        self.visual.load_state_dict({k:state[k] for k in required},strict=True)
        print('Loaded ALL backbone + input_proj + 6-layer encoder weights;',
              len(extra),'unused detection decoder/head keys.')
        del state,payload
        self.requires_grad_(False)
        self.eval()
    def train(self,mode=True):
        super().train(False)
        return self
    @torch.no_grad()
    @torch.autocast(device_type='cuda',dtype=torch.float16)
    def forward(self,images):
        padding,memory = self.visual(nested_tensor_from_tensor_list(images))
        h,w = math.ceil(images.shape[-2]/32),math.ceil(images.shape[-1]/32)
        assert tuple(memory.shape) == (h*w,len(images),256)
        assert not padding.any(), 'Ảnh đã resize cùng size nên mask phải toàn False.'
        return memory.permute(1,2,0).reshape(len(images),256,h,w).contiguous()

## 3. Trích feature và lưu cache

Resize toàn ảnh theo `image_size_hw` trong REC checkpoint, dùng chuẩn hóa như pipeline validation. ROI Align lấy feature từ bbox của người trên feature map stride 32 rồi trung bình theo không gian, cho một vector 256 chiều/người. Pooling cả bbox có thể trộn áo với quần, nền và người đứng cạnh. Với người rất nhỏ, feature map có ít chi tiết để tách màu áo.

Đặt `EXTRACT_FEATURES = False` khi muốn dùng cache đã tạo, hoặc dừng sau cell này để train probe sau.

In [7]:
%pip -q install -U gdown

from pathlib import Path
from google.colab import drive
import gdown

drive.mount("/content/drive")

# Giữ đúng cách viết Unicode của đường dẫn trong notebook.
checkpoint = (
    Path("/content/drive/MyDrive/Computer Vision")
    / "kie\u0302\u0309m chu\u031b\u0301ng"
    / "detr-r50-unc.pth"
)
checkpoint.parent.mkdir(parents=True, exist_ok=True)

downloaded = gdown.download(
    id="1U8Qdl-StW1yaJapnGtAEFO8Ic4X0ubtp",
    output=str(checkpoint),
    quiet=False,
)
if downloaded is None or not checkpoint.is_file():
    raise RuntimeError("Tải thất bại. Kiểm tra quyền truy cập của link Drive.")

print(f"Đã lưu {checkpoint.stat().st_size / 1024**2:.1f} MiB")
print(checkpoint)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Downloading...
From (original): https://drive.google.com/uc?id=1U8Qdl-StW1yaJapnGtAEFO8Ic4X0ubtp
From (redirected): https://drive.google.com/uc?id=1U8Qdl-StW1yaJapnGtAEFO8Ic4X0ubtp&confirm=t&uuid=0369fa8b-d318-445e-9f05-596e29279937
To: /content/drive/MyDrive/Computer Vision/kiểm chứng/detr-r50-unc.pth
100%|██████████| 128M/128M [00:01<00:00, 93.0MB/s]


Đã lưu 122.1 MiB
/content/drive/MyDrive/Computer Vision/kiểm chứng/detr-r50-unc.pth


In [8]:
import torch.nn.functional as F

In [9]:
class RECVisual(nn.Module):
    def __init__(self, detr_path, rec_state):
        super().__init__()
        visual = FrozenDETRFeatures(detr_path).visual
        backbone = list(visual.backbone[0].body.children())
        self.stem = nn.Sequential(*backbone[:-1])
        self.layer4 = backbone[-1]
        self.pos = visual.backbone[1]
        self.proj = visual.input_proj
        self.visual_encoder = visual.transformer.encoder
        for name in ('layer4', 'proj', 'visual_encoder'):
            prefix = name + '.'
            state = {key[len(prefix):]: value for key, value in rec_state.items()
                     if key.startswith(prefix)}
            getattr(self, name).load_state_dict(state, strict=True)
        self.requires_grad_(False).eval()

    def forward(self, image):
        x = self.proj(self.layer4(self.stem(image)))
        nested = nested_tensor_from_tensor_list(x)
        pos = self.pos(nested).to(x.dtype).flatten(2).permute(2, 0, 1)
        tokens = x.flatten(2).permute(2, 0, 1)
        memory = self.visual_encoder(tokens, src_key_padding_mask=nested.mask.flatten(1),
                                     pos=pos)
        return memory.permute(1, 2, 0).reshape_as(x)

@torch.inference_mode()
def extract_color_features(frame):
    rec = torch.load(REC_CHECKPOINT, map_location='cpu', weights_only=True)
    cfg = rec['config']
    if cfg.get('detr_sha256') != sha256(DETR_CHECKPOINT):
        raise ValueError('Checkpoint DETR gốc không khớp SHA256 trong REC checkpoint.')
    if cfg.get('annotation_sha256') != annotation_sha:
        raise ValueError('Annotation không khớp với bộ dữ liệu của REC checkpoint.')

    height, width = map(int, cfg['image_size_hw'])
    visual = RECVisual(DETR_CHECKPOINT, rec['model_state_dict']).to(DEVICE)
    epoch = rec['epoch']
    del rec
    resize = transforms.Compose([transforms.Resize((height, width)), transforms.PILToTensor()])
    mean = torch.tensor([.485, .456, .406], device=DEVICE).view(1, 3, 1, 1)
    std = torch.tensor([.229, .224, .225], device=DEVICE).view(1, 3, 1, 1)

    grouped = list(frame.groupby('image_id', sort=False))
    features, metadata = [], []
    for start in tqdm(range(0, len(grouped), BATCH_IMAGES), desc='Trích feature theo batch ảnh'):
        images, rois, batch_rows = [], [], []
        for batch_index, (_, rows) in enumerate(grouped[start:start + BATCH_IMAGES]):
            first = rows.iloc[0]
            with Image.open(IMAGE_DIR / first.file_name) as image:
                if image.size != (first.width, first.height):
                    raise ValueError(f'Sai kích thước ảnh: {first.file_name}')
                images.append(resize(image.convert('RGB')))
            for row in rows.itertuples(index=False):
                x1, x2 = np.clip([row.x, row.x + row.w], 0, row.width) * width / row.width
                y1, y2 = np.clip([row.y, row.y + row.h], 0, row.height) * height / row.height
                if x2 <= x1 or y2 <= y1:
                    raise ValueError(f'BBox không hợp lệ: {row.review_id}')
                rois.append([batch_index, x1, y1, x2, y2])
                batch_rows.append(dict(review_id=row.review_id, image_id=int(row.image_id),
                                       split=row.split, color=row.color))
        inputs = (torch.stack(images).to(DEVICE).float().div(255) - mean) / std
        with torch.autocast('cuda', dtype=torch.float16, enabled=DEVICE.type == 'cuda'):
            memory = visual(inputs)
        roi_tensor = torch.as_tensor(rois, dtype=torch.float32, device=DEVICE)
        pooled = roi_align(memory.float(), roi_tensor, output_size=(3, 3),
                           spatial_scale=1 / 32, sampling_ratio=2, aligned=True)
        features.append(pooled.mean(dim=(-2, -1)).cpu())
        metadata.extend(batch_rows)

    return dict(features=torch.cat(features), rows=metadata, rec_epoch=int(epoch),
                rec_sha256=sha256(REC_CHECKPOINT), annotation_sha256=annotation_sha,
                labels_sha256=sha256(LABEL_PATH), image_size_hw=[height, width])

if EXTRACT_FEATURES:
    cache = extract_color_features(read_labels())
    temporary = CACHE_PATH.with_suffix('.tmp')
    torch.save(cache, temporary)
    temporary.replace(CACHE_PATH)
    print(f"REC epoch {cache['rec_epoch']} | {tuple(cache['features'].shape)} | {CACHE_PATH}")

Loaded ALL backbone + input_proj + 6-layer encoder weights; 0 unused detection decoder/head keys.


Trích feature theo batch ảnh:   0%|          | 0/69 [00:00<?, ?it/s]

REC epoch 42 | (570, 256) | /content/drive/MyDrive/Computer Vision/kiểm chứng/color_features_auto.pt


## 4. Fit linear probe và đánh giá

Bật `RUN_PROBE = True` rồi chạy cell dưới. Probe dùng logistic regression đa lớp, tương đương một lớp tuyến tính cho điểm từng màu. Chuẩn hóa feature bằng thống kê train. Báo accuracy, balanced accuracy, macro F1 và confusion matrix. Đối chiếu với baseline luôn đoán màu phổ biến nhất trong train.

Điểm trên val đo mức khớp với nhãn màu suy ra từ text. Nhãn này vẫn có thể sai dù đã lọc từ chỉ vị trí. Nếu probe yếu, kết quả chỉ nói về khả năng đọc tuyến tính với cách pooling và tập nhãn này. Kết quả probe kết hợp với H2 giúp khoanh vùng vấn đề, nhưng chưa định vị được chính xác tầng gây lỗi.

In [10]:
def run_linear_probe():
    from sklearn.dummy import DummyClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    cache = torch.load(CACHE_PATH, map_location='cpu', weights_only=True)
    if LABEL_PATH.exists() and cache['labels_sha256'] != sha256(LABEL_PATH):
        raise ValueError('Nhãn đã thay đổi. Trích lại cache trước khi fit probe.')
    if REC_CHECKPOINT.exists() and cache['rec_sha256'] != sha256(REC_CHECKPOINT):
        raise ValueError('REC checkpoint đã đổi. Trích lại cache.')
    rows = pd.DataFrame(cache['rows'])
    x = cache['features'].numpy()
    train = rows['split'].eq('train').to_numpy()
    val = rows['split'].eq('val').to_numpy()
    overlap = set(rows.loc[train, 'image_id']) & set(rows.loc[val, 'image_id'])
    if overlap:
        raise ValueError(f'Train/val trùng {len(overlap)} ảnh. Kiểm tra annotation.')

    counts = pd.crosstab(rows['color'], rows['split']).reindex(
        index=COLORS, columns=['train', 'val'], fill_value=0)
    display(counts)
    labels = counts.index[(counts['train'] >= MIN_TRAIN_PER_COLOR)
                          & (counts['val'] >= MIN_VAL_PER_COLOR)].tolist()
    if len(labels) < 2:
        raise ValueError('Cần ít nhất hai màu đủ mẫu ở cả train và val. Kiểm tra số mẫu mỗi màu ở bảng trên.')
    usable = rows['color'].isin(labels).to_numpy()
    train &= usable
    val &= usable
    y = rows['color'].to_numpy()
    probe = make_pipeline(StandardScaler(), LogisticRegression(
        C=1.0, class_weight='balanced', max_iter=2000, random_state=SEED))
    baseline = DummyClassifier(strategy='most_frequent')
    metrics = {}
    for name, estimator in [('majority', baseline), ('linear_probe', probe)]:
        estimator.fit(x[train], y[train])
        pred = estimator.predict(x[val])
        metrics[name] = dict(
            accuracy=accuracy_score(y[val], pred),
            balanced_accuracy=balanced_accuracy_score(y[val], pred),
            macro_f1=f1_score(y[val], pred, average='macro', labels=labels, zero_division=0),
        )
    pred = probe.predict(x[val])
    matrix = pd.DataFrame(confusion_matrix(y[val], pred, labels=labels),
                          index=labels, columns=labels)
    matrix.index.name, matrix.columns.name = 'Nhãn từ text', 'Dự đoán'
    display(pd.DataFrame(metrics).T.round(4))
    display(matrix)
    print(f"REC epoch {cache['rec_epoch']} | train={train.sum()} | val={val.sum()} | màu={labels}")
    report = dict(rec_epoch=cache['rec_epoch'], rec_sha256=cache['rec_sha256'],
                  labels=labels, train_n=int(train.sum()), val_n=int(val.sum()),
                  counts=counts.to_dict(), metrics=metrics,
                  confusion_matrix=matrix.to_numpy().tolist())
    (OUT / 'probe_metrics_auto.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    predictions = rows.loc[val].copy()
    predictions['prediction'] = pred
    predictions.to_csv(OUT / 'probe_val_predictions_auto.csv', index=False)

if RUN_PROBE:
    run_linear_probe()

split,train,val
color,,
red,70,25
blue,70,25
white,70,25
black,70,25
green,70,25
yellow,70,25


,accuracy,balanced_accuracy,macro_f1
majority,0.1667,0.1667,0.0476
linear_probe,0.3467,0.3467,0.3412


Dự đoán,red,blue,white,black,green,yellow
Nhãn từ text,,,,,,
red,9,5,3,1,3,4
blue,0,5,7,6,5,2
white,2,2,15,2,0,4
black,0,5,5,11,4,0
green,5,7,3,4,3,3
yellow,4,2,3,0,7,9


REC epoch 42 | train=420 | val=150 | màu=['red', 'blue', 'white', 'black', 'green', 'yellow']


In [12]:
import json
from pathlib import Path

samples = json.loads(Path(ANNOTATION_PATH).read_text(encoding="utf-8"))

boxes = pd.DataFrame([
    {
        "review_id": f"{s['image_id']}:{s['ann_id']}",
        "file_name": s["file_name"],
        **dict(zip(("x", "y", "w", "h"), s["bbox"])),
    }
    for s in samples
]).drop_duplicates(subset="review_id")

pred_path = OUT / "probe_val_predictions_auto.csv"
if not pred_path.is_file():
    raise FileNotFoundError(
        "Chưa có file dự đoán. Chạy run_linear_probe() ở cell phía trên trước."
    )

rows = pd.read_csv(pred_path, dtype={"review_id": str})
boxes["review_id"] = boxes["review_id"].astype(str)

rows = rows.merge(boxes, on="review_id", how="left", validate="many_to_one")
assert rows["file_name"].notna().all(), "Có mẫu dự đoán không ghép được bbox."

In [13]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

cache = torch.load(CACHE_PATH, map_location="cpu", weights_only=True)
rows = pd.DataFrame(cache["rows"])
samples = json.loads(Path(ANNOTATION_PATH).read_text(encoding="utf-8"))

# Ghép đúng người trong ảnh với bbox trong JSON.
people = {
    f"{s['image_id']}:{s['ann_id']}": s
    for s in samples
}
if {"image_id", "ann_id"}.issubset(rows.columns):
    keys = rows["image_id"].astype(str) + ":" + rows["ann_id"].astype(str)
elif "review_id" in rows.columns:
    keys = rows["review_id"].astype(str)
else:
    raise ValueError(f"Gửi tui dòng này để sửa khóa ghép: {cache['rows'][0]}")

counts = pd.crosstab(rows["color"], rows["split"]).reindex(
    index=COLORS, columns=["train", "val"], fill_value=0
)
labels = counts.index[
    (counts["train"] >= globals().get("MIN_TRAIN_PER_COLOR", 10))
    & (counts["val"] >= globals().get("MIN_VAL_PER_COLOR", 5))
].tolist()
indices = np.flatnonzero(rows["color"].isin(labels).to_numpy())
selected = rows.iloc[indices]
train = selected["split"].eq("train").to_numpy()
val = selected["split"].eq("val").to_numpy()
y = selected["color"].to_numpy()

if len(labels) < 2:
    raise ValueError("Cache cần ít nhất hai màu đủ mẫu ở cả train và val.")

pixel_features = {"RGB": [], "HSV": []}
for key in keys.iloc[indices]:
    if key not in people:
        raise ValueError(f"Gửi tui dòng này để sửa khóa ghép: {cache['rows'][0]}")

    sample = people[key]
    x, y0, w, h = sample["bbox"]
    with Image.open(Path(IMAGE_DIR) / sample["file_name"]) as image:
        box = (
            max(0, int(np.floor(x))),
            max(0, int(np.floor(y0))),
            min(image.width, int(np.ceil(x + w))),
            min(image.height, int(np.ceil(y0 + h))),
        )
        crop = image.crop(box).convert("RGB")

    for mode, vectors in pixel_features.items():
        pixels = np.asarray(crop.convert(mode))
        hist = [
            np.histogram(pixels[..., channel], bins=16, range=(0, 256))[0]
            / (pixels.shape[0] * pixels.shape[1])
            for channel in range(3)
        ]
        vectors.append(np.concatenate(hist))

features = {
    "DETR": cache["features"].numpy()[indices],
    "RGB": np.stack(pixel_features["RGB"]),
    "HSV": np.stack(pixel_features["HSV"]),
}
scores = {}
for name, X in features.items():
    probe = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=1.0, class_weight="balanced", max_iter=2000,
            random_state=globals().get("SEED", 42),
        ),
    )
    probe.fit(X[train], y[train])
    pred = probe.predict(X[val])
    scores[name] = {
        "accuracy": accuracy_score(y[val], pred),
        "balanced_accuracy": balanced_accuracy_score(y[val], pred),
        "macro_f1": f1_score(y[val], pred, average="macro", zero_division=0),
    }

print(f"Train={train.sum()} | Val={val.sum()}")
display(pd.DataFrame.from_dict(scores, orient="index").round(4))

Train=420 | Val=150


,accuracy,balanced_accuracy,macro_f1
DETR,0.3467,0.3467,0.3412
RGB,0.3867,0.3867,0.3851
HSV,0.4933,0.4933,0.4908


In [14]:
from unittest.mock import patch

import pandas as pd
import torch
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

after = torch.load(CACHE_PATH, map_location="cpu", weights_only=True)

# Dùng đúng bảng nhãn mà cell extract hiện tại của m dùng.
label_rows = read_labels()

if after.get("rec_sha256") is not None:
    if after["rec_sha256"] != sha256(REC_CHECKPOINT):
        raise ValueError("Checkpoint REC đã đổi so với cache. Chạy lại cell extract gốc trước.")

original_forward = RECVisual.forward

def return_encoder_input(module, args, output):
    # args[0] là chuỗi token ngay trước Transformer encoder.
    return args[0]

def forward_using_encoder_input(self, image):
    hook = self.visual_encoder.register_forward_hook(return_encoder_input)
    try:
        return original_forward(self, image)
    finally:
        hook.remove()

# Tái sử dụng pipeline trích feature gốc để giữ đúng ROI Align và pooling.
# Chỉ đổi tensor được đưa vào bước pooling.
with patch.object(RECVisual, "forward", forward_using_encoder_input):
    before = extract_color_features(label_rows)

before_rows = pd.DataFrame(before["rows"])
after_rows = pd.DataFrame(after["rows"])

if not before_rows.equals(after_rows):
    raise ValueError("Mẫu, thứ tự hoặc nhãn khác cache cũ. Chạy lại cell extract gốc trước.")
if before.get("rec_epoch") != after.get("rec_epoch"):
    raise ValueError("Hai lượt trích feature dùng epoch khác nhau.")
if before["features"].shape != after["features"].shape:
    raise ValueError("Hai bộ feature có kích thước khác nhau.")

counts = pd.crosstab(after_rows["color"], after_rows["split"]).reindex(
    index=COLORS, columns=["train", "val"], fill_value=0
)
labels = counts.index[
    (counts["train"] >= globals().get("MIN_TRAIN_PER_COLOR", 10))
    & (counts["val"] >= globals().get("MIN_VAL_PER_COLOR", 5))
].tolist()
if len(labels) < 2:
    raise ValueError("Cần ít nhất hai màu đủ mẫu ở cả train và val.")

usable = after_rows["color"].isin(labels)
train = (usable & after_rows["split"].eq("train")).to_numpy()
val = (usable & after_rows["split"].eq("val")).to_numpy()
y = after_rows["color"].to_numpy()

scores = {}
for name, source in [("Trước encoder", before), ("Sau encoder", after)]:
    X = source["features"].cpu().numpy()
    probe = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=1.0,
            class_weight="balanced",
            max_iter=2000,
            random_state=globals().get("SEED", 42),
        ),
    )
    probe.fit(X[train], y[train])
    pred = probe.predict(X[val])
    scores[name] = {
        "accuracy": accuracy_score(y[val], pred),
        "balanced_accuracy": balanced_accuracy_score(y[val], pred),
        "macro_f1": f1_score(y[val], pred, labels=labels, average="macro", zero_division=0),
    }

print(f"REC epoch {after.get('rec_epoch')} | Train={train.sum()} | Val={val.sum()}")
display(pd.DataFrame(scores).T.round(4))

delta = scores["Sau encoder"]["accuracy"] - scores["Trước encoder"]["accuracy"]
print(f"Chênh lệch accuracy (sau − trước): {delta * 100:+.2f} điểm phần trăm")

Loaded ALL backbone + input_proj + 6-layer encoder weights; 0 unused detection decoder/head keys.


Trích feature theo batch ảnh:   0%|          | 0/69 [00:00<?, ?it/s]

REC epoch 42 | Train=420 | Val=150


,accuracy,balanced_accuracy,macro_f1
Trước encoder,0.5000,0.5000,0.4998
Sau encoder,0.3467,0.3467,0.3412


Chênh lệch accuracy (sau − trước): -15.33 điểm phần trăm


In [15]:
from unittest.mock import patch

import pandas as pd
import torch
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

after = torch.load(CACHE_PATH, map_location="cpu", weights_only=True)

# Dùng đúng bảng nhãn mà cell extract hiện tại của m dùng.
label_rows = read_labels()

if after.get("rec_sha256") is not None:
    if after["rec_sha256"] != sha256(REC_CHECKPOINT):
        raise ValueError("Checkpoint REC đã đổi so với cache. Chạy lại cell extract gốc trước.")

original_forward = RECVisual.forward

def return_encoder_input(module, args, output):
    # args[0] là chuỗi token ngay trước Transformer encoder.
    return args[0]

def forward_using_encoder_input(self, image):
    hook = self.visual_encoder.register_forward_hook(return_encoder_input)
    try:
        return original_forward(self, image)
    finally:
        hook.remove()

# Tái sử dụng pipeline trích feature gốc để giữ đúng ROI Align và pooling.
# Chỉ đổi tensor được đưa vào bước pooling.
with patch.object(RECVisual, "forward", forward_using_encoder_input):
    before = extract_color_features(label_rows)

before_rows = pd.DataFrame(before["rows"])
after_rows = pd.DataFrame(after["rows"])

if not before_rows.equals(after_rows):
    raise ValueError("Mẫu, thứ tự hoặc nhãn khác cache cũ. Chạy lại cell extract gốc trước.")
if before.get("rec_epoch") != after.get("rec_epoch"):
    raise ValueError("Hai lượt trích feature dùng epoch khác nhau.")
if before["features"].shape != after["features"].shape:
    raise ValueError("Hai bộ feature có kích thước khác nhau.")

counts = pd.crosstab(after_rows["color"], after_rows["split"]).reindex(
    index=COLORS, columns=["train", "val"], fill_value=0
)
labels = counts.index[
    (counts["train"] >= globals().get("MIN_TRAIN_PER_COLOR", 10))
    & (counts["val"] >= globals().get("MIN_VAL_PER_COLOR", 5))
].tolist()
if len(labels) < 2:
    raise ValueError("Cần ít nhất hai màu đủ mẫu ở cả train và val.")

usable = after_rows["color"].isin(labels)
train = (usable & after_rows["split"].eq("train")).to_numpy()
val = (usable & after_rows["split"].eq("val")).to_numpy()
y = after_rows["color"].to_numpy()

scores = {}
for name, source in [("Trước encoder", before), ("Sau encoder", after)]:
    X = source["features"].cpu().numpy()
    probe = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=1.0,
            class_weight="balanced",
            max_iter=2000,
            random_state=globals().get("SEED", 42),
        ),
    )
    probe.fit(X[train], y[train])
    pred = probe.predict(X[val])
    scores[name] = {
        "accuracy": accuracy_score(y[val], pred),
        "balanced_accuracy": balanced_accuracy_score(y[val], pred),
        "macro_f1": f1_score(y[val], pred, labels=labels, average="macro", zero_division=0),
    }

print(f"REC epoch {after.get('rec_epoch')} | Train={train.sum()} | Val={val.sum()}")
display(pd.DataFrame(scores).T.round(4))

delta = scores["Sau encoder"]["accuracy"] - scores["Trước encoder"]["accuracy"]
print(f"Chênh lệch accuracy (sau − trước): {delta * 100:+.2f} điểm phần trăm")

Loaded ALL backbone + input_proj + 6-layer encoder weights; 0 unused detection decoder/head keys.


Trích feature theo batch ảnh:   0%|          | 0/69 [00:00<?, ?it/s]

REC epoch 42 | Train=420 | Val=150


,accuracy,balanced_accuracy,macro_f1
Trước encoder,0.5000,0.5000,0.4998
Sau encoder,0.3467,0.3467,0.3412


Chênh lệch accuracy (sau − trước): -15.33 điểm phần trăm


In [16]:
from unittest.mock import patch

import pandas as pd
from torch import nn
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

finetuned_before, finetuned_after = before, after
original_forward = RECVisual.forward

def init_pretrained_visual(self, detr_path, rec_state):
    # Dùng toàn bộ trọng số ảnh từ DETR gốc.
    nn.Module.__init__(self)
    visual = FrozenDETRFeatures(detr_path).visual
    backbone = list(visual.backbone[0].body.children())
    self.stem = nn.Sequential(*backbone[:-1])
    self.layer4 = backbone[-1]
    self.pos = visual.backbone[1]
    self.proj = visual.input_proj
    self.visual_encoder = visual.transformer.encoder
    self.requires_grad_(False).eval()

def forward_pretrained_before_encoder(self, image):
    # Trả token đầu vào encoder về cho bước ROI Align và pooling gốc.
    def use_encoder_input(module, args, output):
        return args[0]

    hook = self.visual_encoder.register_forward_hook(use_encoder_input)
    try:
        return original_forward(self, image)
    finally:
        hook.remove()

label_rows = read_labels()
with patch.object(RECVisual, "__init__", init_pretrained_visual):
    print("Đang đo DETR gốc sau encoder...")
    pretrained_after = extract_color_features(label_rows)

    print("Đang đo DETR gốc trước encoder...")
    with patch.object(RECVisual, "forward", forward_pretrained_before_encoder):
        pretrained_before = extract_color_features(label_rows)

sources = {
    "DETR gốc · trước encoder": pretrained_before,
    "DETR gốc · sau encoder": pretrained_after,
    "REC epoch 42 · trước encoder": finetuned_before,
    "REC epoch 42 · sau encoder": finetuned_after,
}
rows = pd.DataFrame(finetuned_after["rows"])

counts = pd.crosstab(rows["color"], rows["split"]).reindex(
    index=COLORS, columns=["train", "val"], fill_value=0
)
labels = counts.index[
    (counts["train"] >= globals().get("MIN_TRAIN_PER_COLOR", 10))
    & (counts["val"] >= globals().get("MIN_VAL_PER_COLOR", 5))
].tolist()
if len(labels) < 2:
    raise ValueError("Cần ít nhất hai màu đủ mẫu ở cả train và val.")

usable = rows["color"].isin(labels)
train = (usable & rows["split"].eq("train")).to_numpy()
val = (usable & rows["split"].eq("val")).to_numpy()
y = rows["color"].to_numpy()

scores = {}
for name, source in sources.items():
    if not pd.DataFrame(source["rows"]).equals(rows):
        raise ValueError(f"Mẫu, nhãn hoặc thứ tự lệch ở {name}.")

    X = source["features"].cpu().numpy()
    probe = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=1.0, class_weight="balanced", max_iter=2000,
            random_state=globals().get("SEED", 42),
        ),
    )
    probe.fit(X[train], y[train])
    pred = probe.predict(X[val])
    scores[name] = {
        "accuracy": accuracy_score(y[val], pred),
        "macro_f1": f1_score(y[val], pred, labels=labels, average="macro"),
    }

table = pd.DataFrame(scores).T
print(f"Train={train.sum()} | Val={val.sum()}")
display(table.round(4))

for model_name in ("DETR gốc", "REC epoch 42"):
    acc_before = table.loc[f"{model_name} · trước encoder", "accuracy"]
    acc_after = table.loc[f"{model_name} · sau encoder", "accuracy"]
    print(f"{model_name} | Sau − trước: {(acc_after - acc_before) * 100:+.2f} điểm %")

Đang đo DETR gốc sau encoder...
Loaded ALL backbone + input_proj + 6-layer encoder weights; 0 unused detection decoder/head keys.


Trích feature theo batch ảnh:   0%|          | 0/69 [00:00<?, ?it/s]

Đang đo DETR gốc trước encoder...
Loaded ALL backbone + input_proj + 6-layer encoder weights; 0 unused detection decoder/head keys.


Trích feature theo batch ảnh:   0%|          | 0/69 [00:00<?, ?it/s]

Train=420 | Val=150


,accuracy,macro_f1
DETR gốc · trước encoder,0.3200,0.3177
DETR gốc · sau encoder,0.2400,0.2439
REC epoch 42 · trước encoder,0.5000,0.4998
REC epoch 42 · sau encoder,0.3467,0.3412


DETR gốc | Sau − trước: -8.00 điểm %
REC epoch 42 | Sau − trước: -15.33 điểm %


In [17]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

rows = pd.DataFrame(after["rows"])
if not pd.DataFrame(before["rows"]).equals(rows):
    raise ValueError("Mẫu trước và sau encoder không khớp.")

train_idx = np.flatnonzero(train)
val_idx = np.flatnonzero(val)
y = np.asarray(y)
classes = np.unique(y[train_idx])

splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
fit_pos, tune_pos = next(splitter.split(
    train_idx, y[train_idx], groups=rows["image_id"].to_numpy()[train_idx]
))
fit_idx, tune_idx = train_idx[fit_pos], train_idx[tune_pos]

def make_mlp(alpha, seed):
    return MLPClassifier(
        hidden_layer_sizes=(8,),
        activation="relu",
        solver="adam",
        alpha=alpha,
        random_state=seed,
    )

def macro_f1(target, pred):
    return f1_score(
        target, pred, labels=classes, average="macro", zero_division=0
    )

results = []
for name, source, linear_acc in [
    ("Trước encoder", before, 75 / 150),
    ("Sau encoder", after, 52 / 150),
]:
    X = source["features"].cpu().numpy()
    scaler = StandardScaler().fit(X[fit_idx])
    X_fit = scaler.transform(X[fit_idx])
    X_tune = scaler.transform(X[tune_idx])

    best = (-1.0, 0, 0.0)
    for alpha in (1.0, 5.0, 20.0):
        model = make_mlp(alpha, seed=42)
        for epoch in range(1, 151):
            model.partial_fit(X_fit, y[fit_idx], classes=classes)
            if epoch % 5 == 0:
                score = macro_f1(y[tune_idx], model.predict(X_tune))
                # Khi bằng điểm, ưu tiên ít epoch hơn, rồi alpha lớn hơn.
                best = max(best, (score, -epoch, alpha))

    _, negative_epochs, alpha = best
    epochs = -negative_epochs

    scaler = StandardScaler().fit(X[train_idx])
    X_train = scaler.transform(X[train_idx])
    X_val = scaler.transform(X[val_idx])
    runs = []
    for seed in (42, 43, 44, 45, 46):
        model = make_mlp(alpha, seed)
        for _ in range(epochs):
            model.partial_fit(X_train, y[train_idx], classes=classes)
        pred = model.predict(X_val)
        runs.append([
            model.score(X_train, y[train_idx]),
            accuracy_score(y[val_idx], pred),
            macro_f1(y[val_idx], pred),
        ])

    train_acc, val_acc, val_f1 = np.mean(runs, axis=0)
    results.append({
        "Feature": name,
        "Alpha": alpha,
        "Epoch": epochs,
        "Train MLP (%)": 100 * train_acc,
        "Val MLP (%)": 100 * val_acc,
        "Val MLP std (điểm %)": 100 * np.std(np.asarray(runs)[:, 1], ddof=1),
        "Val macro F1": val_f1,
        "Val linear (%)": 100 * linear_acc,
        "MLP − linear (điểm %)": 100 * (val_acc - linear_acc),
    })

print(f"Fit để chọn cấu hình={len(fit_idx)} | Tune={len(tune_idx)}")
print(f"Train cuối={len(train_idx)} | Val cuối={len(val_idx)} | 5 seed")
display(pd.DataFrame(results).set_index("Feature").round(3))

Fit để chọn cấu hình=336 | Tune=84
Train cuối=420 | Val cuối=150 | 5 seed


,Alpha,Epoch,Train MLP (%),Val MLP (%),Val MLP std (điểm %),Val macro F1,Val linear (%),MLP − linear (điểm %)
Feature,,,,,,,,
Trước encoder,1.0,120,67.048,39.600,3.483,0.382,50.000,-10.4
Sau encoder,5.0,145,55.952,32.667,2.906,0.305,34.667,-2.0


In [18]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

frame = read_labels()
counts = pd.crosstab(frame["color"], frame["split"]).reindex(
    index=COLORS, columns=["train", "val"], fill_value=0
)
colors = counts.index[
    (counts["train"] >= MIN_TRAIN_PER_COLOR)
    & (counts["val"] >= MIN_VAL_PER_COLOR)
].tolist()
assert len(colors) >= 2, "Cần ít nhất hai màu đủ mẫu train và val."

frame = frame[
    frame["color"].isin(colors) & frame["split"].isin(["train", "val"])
]
train_ids = set(frame.loc[frame["split"].eq("train"), "image_id"])
val_ids = set(frame.loc[frame["split"].eq("val"), "image_id"])
assert not train_ids & val_ids, "Train và val trùng ảnh."

checkpoint = torch.load(REC_CHECKPOINT, map_location="cpu", weights_only=True)
cfg = checkpoint["config"]
assert cfg["detr_sha256"] == sha256(DETR_CHECKPOINT), "Sai checkpoint DETR."
assert cfg["annotation_sha256"] == annotation_sha, "Sai annotation."
height, width = map(int, cfg["image_size_hw"])
visual = RECVisual(DETR_CHECKPOINT, checkpoint["model_state_dict"]).to(DEVICE)
print(f"REC epoch {checkpoint['epoch']} | Resize {height} × {width}")
del checkpoint

resize = transforms.Compose([
    transforms.Resize((height, width)),
    transforms.PILToTensor(),
])
mean = torch.tensor([.485, .456, .406], device=DEVICE).view(1, 3, 1, 1)
std = torch.tensor([.229, .224, .225], device=DEVICE).view(1, 3, 1, 1)
feature_batches = {
    "layer4_2048": [],
    "sau_proj_256": [],
    "sau_encoder_256": [],
}
metadata = []
groups = list(frame.groupby("image_id", sort=False))

with torch.inference_mode():
    for start in tqdm(range(0, len(groups), BATCH_IMAGES), desc="So sánh 3 điểm lấy feature"):
        images, rois = [], []
        for batch_idx, (_, rows) in enumerate(groups[start:start + BATCH_IMAGES]):
            first = rows.iloc[0]
            with Image.open(IMAGE_DIR / first.file_name) as image:
                assert image.size == (first.width, first.height), first.file_name
                images.append(resize(image.convert("RGB")))

            for row in rows.itertuples(index=False):
                x1, x2 = np.clip([row.x, row.x + row.w], 0, row.width) * width / row.width
                y1, y2 = np.clip([row.y, row.y + row.h], 0, row.height) * height / row.height
                assert x2 > x1 and y2 > y1, f"BBox lỗi: {row.review_id}"
                rois.append([batch_idx, x1, y1, x2, y2])
                metadata.append((row.split, row.color))

        inputs = (torch.stack(images).to(DEVICE).float() / 255 - mean) / std
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            layer4 = visual.layer4(visual.stem(inputs))
            projected = visual.proj(layer4)
            nested = nested_tensor_from_tensor_list(projected)
            pos = visual.pos(nested).to(projected.dtype).flatten(2).permute(2, 0, 1)
            memory = visual.visual_encoder(
                projected.flatten(2).permute(2, 0, 1),
                src_key_padding_mask=nested.mask.flatten(1),
                pos=pos,
            )
            encoded = memory.permute(1, 2, 0).reshape_as(projected)

        roi_tensor = torch.as_tensor(rois, dtype=torch.float32, device=DEVICE)
        for name, fmap in zip(feature_batches, (layer4, projected, encoded)):
            pooled = roi_align(
                fmap.float(), roi_tensor, output_size=(3, 3),
                spatial_scale=1 / 32, sampling_ratio=2, aligned=True,
            )
            feature_batches[name].append(pooled.mean(dim=(-2, -1)).cpu())

rows = pd.DataFrame(metadata, columns=["split", "color"])
train = rows["split"].eq("train").to_numpy()
val = rows["split"].eq("val").to_numpy()
y = rows["color"].to_numpy()

def score_probe(estimator, x):
    estimator.fit(x[train], y[train])
    pred = estimator.predict(x[val])
    return {
        "accuracy": accuracy_score(y[val], pred),
        "balanced_accuracy": balanced_accuracy_score(y[val], pred),
        "macro_f1": f1_score(y[val], pred, labels=colors, average="macro", zero_division=0),
    }

results = {
    "majority": score_probe(
        DummyClassifier(strategy="most_frequent"), np.zeros((len(y), 1))
    )
}
for name, batches in feature_batches.items():
    x = torch.cat(batches).numpy()
    probe = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=1.0, class_weight="balanced", max_iter=2000, random_state=SEED),
    )
    results[name] = score_probe(probe, x)

print(f"Train: {train.sum()} | Val: {val.sum()} | Màu: {colors}")
display(pd.DataFrame(results).T.round(4))
del visual

Loaded ALL backbone + input_proj + 6-layer encoder weights; 0 unused detection decoder/head keys.
REC epoch 42 | Resize 488 × 588


So sánh 3 điểm lấy feature:   0%|          | 0/69 [00:00<?, ?it/s]

Train: 420 | Val: 150 | Màu: ['red', 'blue', 'white', 'black', 'green', 'yellow']


,accuracy,balanced_accuracy,macro_f1
majority,0.1667,0.1667,0.0476
layer4_2048,0.4800,0.4800,0.4694
sau_proj_256,0.5000,0.5000,0.4998
sau_encoder_256,0.3467,0.3467,0.3412
